In [11]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
import scipy
import sklearn
from sklearn.svm import SVC
from scipy.stats import pearsonr
from sklearn import datasets, linear_model
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import (
    train_test_split, 
    StratifiedKFold, cross_val_score,  
    RepeatedStratifiedKFold, 
    RandomizedSearchCV,
    train_test_split, 
    KFold
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    f1_score
)
from sklearn.feature_selection import SelectFromModel
from sklearn.compose import ColumnTransformer
from scipy.stats import loguniform
from scipy import sparse
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import make_scorer, balanced_accuracy_score, f1_score

#!pip install xlrd 
#!pip install category_encoders
import category_encoders as ce



In [12]:
df = pd.read_excel("TrainDataset2025.xls")
df.head()

,ID,pCR (outcome),RelapseFreeSurvival (outcome),Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002174,1,144.0,41.0,0,0,0,1,3,3,...,0.517172,0.375126,3.325332,0.002314,3880771.500,473.464852,0.000768,0.182615,0.030508,0.000758
1,TRG002178,0,142.0,39.0,1,1,0,0,3,3,...,0.444391,0.444391,3.032144,0.005612,2372009.744,59.459710,0.004383,0.032012,0.001006,0.003685
2,TRG002204,1,135.0,31.0,0,0,0,1,2,1,...,0.534549,0.534549,2.485848,0.006752,1540027.421,33.935384,0.007584,0.024062,0.000529,0.006447
3,TRG002206,0,12.0,35.0,0,0,0,1,3,3,...,0.506185,0.506185,2.606255,0.003755,6936740.794,46.859265,0.005424,0.013707,0.000178,0.004543
4,TRG002210,0,109.0,61.0,1,0,0,0,2,1,...,0.462282,0.462282,2.809279,0.006521,1265399.054,39.621023,0.006585,0.034148,0.001083,0.005626


In [13]:
df.info()
df.describe()
df.replace(999, pd.NA, inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Columns: 121 entries, ID to original_ngtdm_Strength
dtypes: float64(108), int64(12), object(1)
memory usage: 378.3+ KB


In [14]:
# Get columns with missing values
missing_cols = df.columns[df.isna().any()]

# Missing data summary before droppig "pCR" missing values rows and "RelapseFreeSurvival" column
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df[col].isna().sum() for col in missing_cols],
    'Dtype': [df[col].dtype for col in missing_cols]
})

summary

,Column,MissingCount,Dtype
0,pCR (outcome),5,object
1,PgR,1,object
2,HER2,1,object
3,TrippleNegative,1,object
4,ChemoGrade,3,object
5,Proliferation,2,object
6,HistologyType,3,object
7,LNStatus,1,object
8,Gene,88,object


In [15]:
df_dropped =  df.dropna(subset=['pCR (outcome)'])
df_dropped = df_dropped.drop(["RelapseFreeSurvival (outcome)", "ID"], axis=1)
df_dropped.shape

(395, 119)

In [16]:
missing_cols = df_dropped.columns[df_dropped.isna().any()]
# Missing data summary
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df_dropped[col].isna().sum() for col in missing_cols],
    'Dtype': [df_dropped[col].dtype for col in missing_cols]
})

print(summary)

            Column  MissingCount   Dtype
0              PgR             1  object
1             HER2             1  object
2  TrippleNegative             1  object
3       ChemoGrade             3  object
4    Proliferation             2  object
5    HistologyType             3  object
6         LNStatus             1  object
7             Gene            85  object


In [17]:
# Iterative imputation

df_imputed = df_dropped.copy()

## Select categorical columns
cat_columns = df_imputed.select_dtypes(include=['object', 'category', 'string']).columns
cat_cols_to_encode = cat_columns.drop('pCR (outcome)')

encoder = ce.OrdinalEncoder(handle_missing='return_nan') 


## Encode categorical features
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])


## Imputation with IterativeImputer
y_target = df_imputed['pCR (outcome)']
X_features = df_imputed.drop(columns=['pCR (outcome)'])

# Iterative Imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Convert back to DataFrame
X_imputed_df = pd.DataFrame(X_imputed,
                            columns=X_features.columns,
                            index=X_features.index)

# Round encoded categorical columns back to integers
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

    
# Inverse transform encoded categorical columns
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])


# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]
print("\n--- Missing Value Check ---")
print(final_df.isna().sum().sum())



y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)"], axis=1)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


# Train / test split (hold out 30% for final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

keep_cats = ["Gene", "HER2", "ER"] 
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop"
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop"
)




--- Missing Value Check ---
0


In [18]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
# Select features whose RandomForest importance is above a given percentile.
# Works as a drop-in transformer inside a Pipeline.
class RFPercentileSelector(BaseEstimator, TransformerMixin):
    
    def __init__(
        self,
        percentile=50,
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ):
        self.percentile = percentile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.class_weight = class_weight

    def fit(self, X, y):
        # Train RF on the *preprocessed* matrix X
        self.rf_ = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            class_weight=self.class_weight,
        )
        self.rf_.fit(X, y)

        importances = self.rf_.feature_importances_
        self.threshold_ = np.percentile(importances, self.percentile)
        self.mask_ = importances >= self.threshold_

        # For compatibility with some sklearn utilities
        self.feature_importances_ = importances
        return self

    def transform(self, X):
        return X[:, self.mask_]


In [19]:
# Feature selectors
rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=20,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ),
    threshold="median", # mean, better result using median
)

logreg_l1_selector = SelectFromModel( # poor result
    LogisticRegression(
        penalty="l1",
        solver="saga",
        class_weight="balanced",
        random_state=42,
        max_iter=5000,
    ),
    threshold="median"  # or "mean"
)


from sklearn.feature_selection import RFE
from sklearn.svm import LinearSVC

rfe_selector = RFE(
    estimator=LinearSVC(
        penalty="l2",
        class_weight="balanced",
        random_state=42,
    ),
    n_features_to_select=45,   # best performance at 45 
    step=0.1
)

In [10]:
# SVM using default threshold - unstable
## Features = [ER/HER2/Gene one-hot] + [RF-selected other features]
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector), 
    ])),
])

# Pipelinge
svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)


param_dist = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)


search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_svm = search.best_estimator_

# Evaluation of Model
# Fit best pipeline on full training set
best_svm.fit(X_train, y_train)

# Decision scores on test set
scores_test = best_svm.decision_function(X_test)  # continuous margins
y_pred_default = best_svm.predict(X_test)

# Metrics with default decision threshold
roc_auc = roc_auc_score(y_test, scores_test)
pr_auc = average_precision_score(y_test, scores_test)  # PR AUC
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix (default threshold):\n", confusion_matrix(y_test, y_pred_default))
print("Classification report (default threshold):\n", classification_report(y_test, y_pred_default))

Fitting 25 folds for each of 50 candidates, totalling 1250 fits
Best params: {'svm__C': np.float64(361.2478500429091), 'svm__class_weight': 'balanced', 'svm__gamma': np.float64(0.00037961668958008145)}
Best CV ROC AUC: 0.7007319495163047
Test ROC AUC: 0.7663829787234042
Test PR AUC: 0.5079372029645486
Test balanced accuracy: 0.7204255319148936
Confusion matrix (default threshold):
 [[64 30]
 [ 6 19]]
Classification report (default threshold):
               precision    recall  f1-score   support

           0       0.91      0.68      0.78        94
           1       0.39      0.76      0.51        25

    accuracy                           0.70       119
   macro avg       0.65      0.72      0.65       119
weighted avg       0.80      0.70      0.72       119



In [20]:
# SVM with threshold tuning
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])


# Inner train/validation split for threshold tuning
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)

# Pipeline 
svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

param_dist_svm = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

bal_acc_scorer = make_scorer(balanced_accuracy_score)
f1_pos_scorer = make_scorer(f1_score, pos_label=1)

search_svm = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist_svm,
    n_iter=50,
    cv=cv,
    scoring=bal_acc_scorer, # scoring by balance accuracy gives best result
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for SVM on inner training data...")
search_svm.fit(X_train_inner, y_train_inner)

print("\nBest SVM params:", search_svm.best_params_)
print("Best inner CV ROC AUC:", search_svm.best_score_)

best_svm_inner = search_svm.best_estimator_

# Threshold tuning on validation set (Youden / balanced acc)
# Find the threshold that maximises balanced accuracy using the ROC curve.
def tune_threshold_from_scores(scores, y_true):

    scores = np.asarray(scores)
    y_true = np.asarray(y_true)

    # fpr, tpr, thresholds
    fpr, tpr, thresholds = roc_curve(y_true, scores)

    # balanced accuracy = (TPR + TNR) / 2 = (TPR + (1 - FPR)) / 2
    bal_acc = (tpr + (1 - fpr)) / 2.0

    # index of maximum balanced accuracy
    idx = np.argmax(bal_acc)
    best_thr = thresholds[idx]
    best_bal = bal_acc[idx]

    return best_thr, best_bal

# Find the threshold that maximises F1-score for the positive class.
def tune_threshold_for_f1(scores, y_true, positive_label=1):
    
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)

    # Generate all possible score cutpoints that change classification
    fpr, tpr, thresholds = roc_curve(y_true, scores)

    best_thr = thresholds[0]
    best_f1 = -1.0

    # Iterate over all meaningful threshold candidates
    for thr in thresholds:
        y_pred = (scores >= thr).astype(int)
        f1 = f1_score(y_true, y_pred, pos_label=positive_label)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return best_thr, best_f1


    

scores_val = best_svm_inner.decision_function(X_val)
best_thr, best_bal_val = tune_threshold_from_scores(scores_val, y_val)

print(f"\nBest SVM threshold on validation (by balanced accuracy): {best_thr:.4f}")
print(f"Validation balanced accuracy at best threshold: {best_bal_val:.4f}")

y_val_pred_tuned = (scores_val >= best_thr).astype(int)

print("\n--- Validation metrics at tuned threshold ---")
print("ROC AUC (val):", roc_auc_score(y_val, scores_val))
print("PR AUC (val):", average_precision_score(y_val, scores_val))
print("Balanced accuracy (val):", balanced_accuracy_score(y_val, y_val_pred_tuned))
print("Confusion matrix (val):\n", confusion_matrix(y_val, y_val_pred_tuned))
print("Classification report (val):\n", classification_report(y_val, y_val_pred_tuned))

# Refitting SVM on full training data and evaluating on test data
best_svm_final = search_svm.best_estimator_
best_svm_final.fit(X_train, y_train)   # X_train = inner + val

scores_test = best_svm_final.decision_function(X_test)

# Default threshold
y_test_pred_default = best_svm_final.predict(X_test)

# Tuned threshold from validation
y_test_pred_tuned = (scores_test >= best_thr).astype(int)

print("\n=== SVM Test Performance (default decision threshold) ===")
print("Test ROC AUC:", roc_auc_score(y_test, scores_test))
print("Test PR AUC:", average_precision_score(y_test, scores_test))
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred_default))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred_default))
print("Classification report:\n", classification_report(y_test, y_test_pred_default))

print("\n=== SVM Test Performance (tuned threshold from validation) ===")
print("Threshold used:", best_thr)
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred_tuned))
print("Confusion matrix (tuned):\n", confusion_matrix(y_test, y_test_pred_tuned))
print("Classification report (tuned):\n", classification_report(y_test, y_test_pred_tuned))



Fitting RandomizedSearchCV for SVM on inner training data...
Fitting 25 folds for each of 50 candidates, totalling 1250 fits


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0


Best SVM params: {'svm__C': np.float64(43.00001586162607), 'svm__class_weight': 'balanced', 'svm__gamma': np.float64(0.002933870049164396)}
Best inner CV ROC AUC: 0.6290924369747899

Best SVM threshold on validation (by balanced accuracy): -0.3367
Validation balanced accuracy at best threshold: 0.6591

--- Validation metrics at tuned threshold ---
ROC AUC (val): 0.6837121212121212
PR AUC (val): 0.3498626693288195
Balanced accuracy (val): 0.6590909090909092
Confusion matrix (val):
 [[25 19]
 [ 3  9]]
Classification report (val):
               precision    recall  f1-score   support

           0       0.89      0.57      0.69        44
           1       0.32      0.75      0.45        12

    accuracy                           0.61        56
   macro avg       0.61      0.66      0.57        56
weighted avg       0.77      0.61      0.64        56


=== SVM Test Performance (default decision threshold) ===
Test ROC AUC: 0.7612765957446809
Test PR AUC: 0.5377965228522424
Test balanced

In [35]:
import joblib 

artifact = {
    "model": best_svm_final,
    "threshold": float(best_thr), 
}

joblib_file = "svm_classification_model.joblib"
joblib.dump(artifact, joblib_file)

print(f"Saved model + threshold to {joblib_file}")


Saved model + threshold to svm_classification_model.joblib


In [12]:
# ANN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])


# Fit preprocessing + feature selection
full_features.fit(X_train, y_train)

X_train_proc = full_features.transform(X_train)
X_test_proc = full_features.transform(X_test)

# Convert sparse to dense for Keras
if sparse.issparse(X_train_proc):
    X_train_proc = X_train_proc.toarray()
if sparse.issparse(X_test_proc):
    X_test_proc = X_test_proc.toarray()

print("Processed train shape:", X_train_proc.shape)
print("Processed test shape:", X_test_proc.shape)

# Building ANN model in TensorFlow
input_dim = X_train_proc.shape[1]

from tensorflow.keras import layers, regularizers

def build_ann_model(input_dim: int) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        
        # Block 1
        layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4),
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Block 2
        layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4),
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        # Block 3
        layers.Dense(
            64,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4),
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Block 3
        layers.Dense(
            64,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4),
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        
        # Output
        layers.Dense(1, activation="sigmoid"),  # binary classification
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
            keras.metrics.BinaryAccuracy(name="accuracy"),
        ],
    )
    return model

model = build_ann_model(input_dim)

# Class weights + callbacks
classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weight_dict = {cls: w for cls, w in zip(classes, class_weights_array)}
print("\nClass weights:", class_weight_dict)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=15,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1,
)

# Train ANN 
history = model.fit(
    X_train_proc,
    y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weight_dict,
    verbose=1,
)

# Evaluation on test set
# Probabilities (for class 1)
proba_test = model.predict(X_test_proc).ravel()
y_pred_default = (proba_test >= 0.5).astype(int)  # default threshold 0.5

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("\n--- ANN Test Performance (default threshold 0.5) ---")
print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default))
print("Classification report:\n", classification_report(y_test, y_pred_default))

2025-12-09 09:22:41.667595: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-09 09:22:41.684077: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 09:22:42.222541: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-09 09:22:43.986482: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

Processed train shape: (276, 68)
Processed test shape: (119, 68)

Class weights: {np.int64(0): np.float64(0.6359447004608295), np.int64(1): np.float64(2.3389830508474576)}
Epoch 1/200


2025-12-09 09:22:45.048749: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.4909 - auc: 0.5608 - loss: 0.9039 - pr_auc: 0.2270 - val_accuracy: 0.7143 - val_auc: 0.6586 - val_loss: 0.6707 - val_pr_auc: 0.4011 - learning_rate: 0.0010
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5682 - auc: 0.5732 - loss: 0.8893 - pr_auc: 0.2494 - val_accuracy: 0.7321 - val_auc: 0.6539 - val_loss: 0.6865 - val_pr_auc: 0.4127 - learning_rate: 0.0010
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4727 - auc: 0.4875 - loss: 0.9202 - pr_auc: 0.2223 - val_accuracy: 0.6607 - val_auc: 0.6758 - val_loss: 0.7005 - val_pr_auc: 0.4956 - learning_rate: 0.0010
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5455 - auc: 0.5914 - loss: 0.8281 - pr_auc: 0.2899 - val_accuracy: 0.6607 - val_auc: 0.7188 - val_loss: 0.7165 - val_pr_auc: 0.4768 - learning_rate: 0.0010
Epoch 5/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5955 - auc: 0.6971 - loss: 0.7011 - pr_auc: 0.3360 - val_accuracy: 0.

In [36]:
print(cat_columns)

Index(['pCR (outcome)', 'PgR', 'HER2', 'TrippleNegative', 'ChemoGrade',
       'Proliferation', 'HistologyType', 'LNStatus', 'Gene'],
      dtype='object')

In [15]:
from sklearn import set_config
from sklearn.utils import estimator_html_repr

html = estimator_html_repr(best_svm_final)

with open("svm_pipeline.html", "w") as f:
    f.write(html)